# Actualizaciones en la presentación de información referente a casos de COVID-19 en México.


El paquete de datos abiertos que la Dirección General de Epidemiología publica el día *8 de julio del 2024*, incluyó cambios respecto a los paquetes anteriores; en especificó incluyo variables que describen el tipo de virus respiratorio identificado mediante prueba de PCR, así como posibles coinfecciones; de esta manera, ya no sólo se muestra información relacionada a pacientes de COVID-19, sino de todos los Virus Respiratorios que se vigilan mediante el Sistema de Vigilancia Epidemiológica de las Enfermedades Respiratorias Virales (SISVER).

A continuación, se detallan los cambios en el paquete de datos abiertos correspondientes al 8 de julio:

Se elimina la variable RESULTADO_LAB, la cual es reemplazada por las variables RESULTADO_PCR y RESULTADO_PCR_COINFECCION

Se agrega la variable RESULTADO_PCR la cual identifica el resultado de la muestra del paciente a los diferentes virus respiratorios que se analizan en los laboratorios de la Red Nacional de Laboratorios de Vigilancia Epidemiológica (INDRE, LESP y LAVE) y laboratorios privados avalados por el InDRE cuyos resultados son registrados en SISVER.

Se agrega la variable RESULTADO_PCR_COINFECCION la cual permite identificar una coinfección (infección simultanea) encontrada en la muestra del paciente a los diferentes virus respiratorios que se analizan en los laboratorios de la Red Nacional de Laboratorios de Vigilancia Epidemiológica (INDRE, LESP y LAVE) y laboratorios privados avalados por el InDRE cuyos resultados son registrados en SISVER.

Se renombra la variable CLASIFICACIÓN FINAL a CLASIFICACIÓN_FINAL_COVID, esta variable permite identificar si el caso es positivo a COVID-19.

Se agrega la variable CLASIFICACION_FINAL_FLU, esta variable permite identificar si se trata de un caso de Influenza.

In [1]:
from src.database import get_database

db = get_database()

In [13]:
collections = sorted(db.list_collection_names())

print(collections)
for collection in collections:
    print(f"Collection: {collection}")
    print(f"Columns: {db[collection].find_one().keys()}")
    print(f"\nCount keys: {len(db[collection].find_one().keys())}\n")

['raw_2020_data', 'raw_2021_data', 'raw_2022_data', 'raw_2023_data', 'raw_2024_data', 'raw_2025_data', 'raw_2026_data']
Collection: raw_2020_data
Columns: dict_keys(['_id', 'FECHA_ACTUALIZACION', 'ID_REGISTRO', 'ORIGEN', 'SECTOR', 'ENTIDAD_UM', 'SEXO', 'ENTIDAD_NAC', 'ENTIDAD_RES', 'MUNICIPIO_RES', 'TIPO_PACIENTE', 'FECHA_INGRESO', 'FECHA_SINTOMAS', 'FECHA_DEF', 'INTUBADO', 'NEUMONIA', 'EDAD', 'NACIONALIDAD', 'EMBARAZO', 'HABLA_LENGUA_INDIG', 'INDIGENA', 'DIABETES', 'EPOC', 'ASMA', 'INMUSUPR', 'HIPERTENSION', 'OTRA_COM', 'CARDIOVASCULAR', 'OBESIDAD', 'RENAL_CRONICA', 'TABAQUISMO', 'OTRO_CASO', 'TOMA_MUESTRA_LAB', 'RESULTADO_LAB', 'TOMA_MUESTRA_ANTIGENO', 'RESULTADO_ANTIGENO', 'CLASIFICACION_FINAL', 'MIGRANTE', 'PAIS_NACIONALIDAD', 'PAIS_ORIGEN', 'UCI'])

Count keys: 41

Collection: raw_2021_data
Columns: dict_keys(['_id', 'FECHA_ACTUALIZACION', 'ID_REGISTRO', 'ORIGEN', 'SECTOR', 'ENTIDAD_UM', 'SEXO', 'ENTIDAD_NAC', 'ENTIDAD_RES', 'MUNICIPIO_RES', 'TIPO_PACIENTE', 'FECHA_INGRESO', 'FECH

In [ ]:
import re
from pymongo import UpdateOne
from src.database import get_database

db = get_database()

def standardize_document(doc, year_collection):
    doc_estandarizado = {}
    campos_inconsistentes = [
        'RESULTADO_LAB', 'CLASIFICACION_FINAL', 
        'RESULTADO_PCR', 'RESULTADO_PCR_COINFECCION', 
        'CLASIFICACION_FINAL_COVID', 'CLASIFICACION_FINAL_FLU'
    ]
    for clave, valor in doc.items():
        if clave not in campos_inconsistentes and clave != '_id':
            doc_estandarizado[clave] = valor
            
    doc_estandarizado['YEAR_ORIGIN'] = year_collection
    
    if year_collection < 2024:
        doc_estandarizado['RESULTADO_PCR'] = doc.get('RESULTADO_LAB')
        doc_estandarizado['RESULTADO_PCR_COINFECCION'] = '997' 
        doc_estandarizado['CLASIFICACION_FINAL_COVID'] = doc.get('CLASIFICACION_FINAL')
        doc_estandarizado['CLASIFICACION_FINAL_FLU'] = '6'     
    else:
        doc_estandarizado['RESULTADO_PCR'] = doc.get('RESULTADO_PCR')
        doc_estandarizado['RESULTADO_PCR_COINFECCION'] = doc.get('RESULTADO_PCR_COINFECCION')
        doc_estandarizado['CLASIFICACION_FINAL_COVID'] = doc.get('CLASIFICACION_FINAL_COVID')
        doc_estandarizado['CLASIFICACION_FINAL_FLU'] = doc.get('CLASIFICACION_FINAL_FLU')
        
    return doc_estandarizado


def procesar_historico_a_master(chunk=50000):
    coleccion_destino = db["covid_unified_data"]
    colecciones_crudas = sorted(db.list_collection_names())
    
    for nombre_coll in colecciones_crudas:
        if not nombre_coll.startswith("raw_") or not nombre_coll.endswith("_data"):
            continue
            
        year = int(re.search(r'\d{4}', nombre_coll).group())
        print(f"\n Iniciando migración controlada de: {nombre_coll}...")
        
        cursor = db[nombre_coll].find().batch_size(chunk)
        
        operaciones_lote = []
        contador_procesados = 0
        
        for doc in cursor:
            doc_limpio = standardize_document(doc, year)
            id_registro = doc_limpio.get("ID_REGISTRO")
            
            operacion = UpdateOne(
                {"ID_REGISTRO": id_registro},
                {"$set": doc_limpio},
                upsert=True
            )
            operaciones_lote.append(operacion)
            
            if len(operaciones_lote) == chunk:
                resultado = coleccion_destino.bulk_write(operaciones_lote, ordered=False)
                contador_procesados += len(operaciones_lote)
                
                print(f"Procesados: {contador_procesados} | Nuevos creados: {resultado.upserted_count} | Actualizados: {resultado.modified_count}")
                operaciones_lote = []
                
        if operaciones_lote:
            resultado = coleccion_destino.bulk_write(operaciones_lote, ordered=False)
            contador_procesados += len(operaciones_lote)
            print(f"Procesados: {contador_procesados} | Nuevos creados: {resultado.upserted_count} | Actualizados: {resultado.modified_count}")

        print(f"¡Finalizado con éxito! {nombre_coll} completamente integrado.")

if __name__ == "__main__":
    db["covid_unified_data"].create_index("ID_REGISTRO", unique=True)
    
    procesar_historico_a_master()



 Iniciando migración controlada de: raw_2020_data...
Procesados: 50000 | Nuevos creados: 50000 | Actualizados: 0
Procesados: 100000 | Nuevos creados: 50000 | Actualizados: 0
Procesados: 150000 | Nuevos creados: 50000 | Actualizados: 0
Procesados: 200000 | Nuevos creados: 50000 | Actualizados: 0
Procesados: 250000 | Nuevos creados: 50000 | Actualizados: 0
Procesados: 300000 | Nuevos creados: 50000 | Actualizados: 0
Procesados: 350000 | Nuevos creados: 50000 | Actualizados: 0
Procesados: 400000 | Nuevos creados: 50000 | Actualizados: 0
Procesados: 450000 | Nuevos creados: 50000 | Actualizados: 0
Procesados: 500000 | Nuevos creados: 50000 | Actualizados: 0
Procesados: 550000 | Nuevos creados: 50000 | Actualizados: 0
Procesados: 600000 | Nuevos creados: 50000 | Actualizados: 0
Procesados: 650000 | Nuevos creados: 50000 | Actualizados: 0
Procesados: 700000 | Nuevos creados: 50000 | Actualizados: 0
Procesados: 750000 | Nuevos creados: 50000 | Actualizados: 0
Procesados: 800000 | Nuevos crea

1h 

In [16]:
collections = sorted(db.list_collection_names())

print(collections)
for collection in collections:
    print(f"Collection: {collection}")
    print(f"Columns: {db[collection].find_one().keys()}")
    print(f"\nCount keys: {len(db[collection].find_one().keys())}\n")

['covid_unified_data', 'raw_2020_data', 'raw_2021_data', 'raw_2022_data', 'raw_2023_data', 'raw_2024_data', 'raw_2025_data', 'raw_2026_data']
Collection: covid_unified_data
Columns: dict_keys(['_id', 'ID_REGISTRO', 'ASMA', 'CARDIOVASCULAR', 'CLASIFICACION_FINAL_COVID', 'CLASIFICACION_FINAL_FLU', 'DIABETES', 'EDAD', 'EMBARAZO', 'ENTIDAD_NAC', 'ENTIDAD_RES', 'ENTIDAD_UM', 'EPOC', 'FECHA_ACTUALIZACION', 'FECHA_DEF', 'FECHA_INGRESO', 'FECHA_SINTOMAS', 'HABLA_LENGUA_INDIG', 'HIPERTENSION', 'INDIGENA', 'INMUSUPR', 'INTUBADO', 'MIGRANTE', 'MUNICIPIO_RES', 'NACIONALIDAD', 'NEUMONIA', 'OBESIDAD', 'ORIGEN', 'OTRA_COM', 'OTRO_CASO', 'PAIS_NACIONALIDAD', 'PAIS_ORIGEN', 'RENAL_CRONICA', 'RESULTADO_ANTIGENO', 'RESULTADO_PCR', 'RESULTADO_PCR_COINFECCION', 'SECTOR', 'SEXO', 'TABAQUISMO', 'TIPO_PACIENTE', 'TOMA_MUESTRA_ANTIGENO', 'TOMA_MUESTRA_LAB', 'UCI', 'YEAR_ORIGIN'])

Count keys: 44

Collection: raw_2020_data
Columns: dict_keys(['_id', 'FECHA_ACTUALIZACION', 'ID_REGISTRO', 'ORIGEN', 'SECTOR', 'ENT

In [ ]:
print("Total data in Mexico")

print(db.covid_unified_data.count_documents({}))

print("Total COVID-19 cases in CDMX")

print(db.covid_unified_data.count_documents({"ENTIDAD_RES": "09"}))

Total data in Mexico
20769765
Total COVID-19 cases in CDMX
6434347


In [21]:

print(db.covid_unified_data.count_documents({"ENTIDAD_UM": "09"}))

7327689


In [23]:
print(db.covid_unified_data.count_documents({"$or": [{"ENTIDAD_RES": "09"}, {"ENTIDAD_NAC": "09"}]}))


7299904


In [5]:
print(db.covid_unified_data.count_documents({"$or": [{"ENTIDAD_RES": "09"}, {"ENTIDAD_NAC": "09"}]}))


7299605


# Mexico Data Collection 

In [8]:
"""Create the new collection with only CDMX data"""

pipeline = [
    {
        "$match": {"$or": [{"ENTIDAD_RES": "09"}, {"ENTIDAD_NAC": "09"}]}
    },
    {
        "$out": "mexico_city_covid_data"
    }
]

db.covid_unified_data.aggregate(pipeline)
print("Collection 'mexico_city_covid_data' created successfully.")


Collection 'mexico_city_covid_data' created successfully.


In [9]:
print(sorted(db.list_collection_names()))
getIndexes = db.covid_unified_data.index_information()
print(getIndexes)

['covid_unified_data', 'mexico_city_covid_data', 'raw_2020_data', 'raw_2021_data', 'raw_2022_data', 'raw_2023_data', 'raw_2024_data', 'raw_2025_data', 'raw_2026_data']
{'_id_': {'v': 2, 'key': [('_id', 1)]}, 'ID_REGISTRO_1': {'v': 2, 'key': [('ID_REGISTRO', 1)], 'unique': True}}


In [10]:
print(db.mexico_city_covid_data.count_documents({}))

7299605


# Enrich mexico_city_covid_data collection with ENTIDAD_UM 

In [2]:
print(db.list_collection_names())

['raw_2023_data', 'mexico_city_covid_data', 'covid_unified_data', 'raw_2026_data', 'raw_2025_data', 'raw_2022_data', 'raw_2024_data', 'raw_2021_data', 'raw_2020_data']


In [5]:
print(db.covid_unified_data.count_documents({"$or": [{"ENTIDAD_RES": "09"}, {"ENTIDAD_NAC": "09"}]}))
print(db.mexico_city_covid_data.count_documents({}))

7299605
7299605


In [6]:
print(db.covid_unified_data.count_documents({"ENTIDAD_UM": "09"}))

7327356


In [7]:
consulta = {
    "$and": [
        {"ENTIDAD_UM": "09"},
        {"ENTIDAD_RES": {"$ne": "09"}},
        {"ENTIDAD_NAC": {"$ne": "09"}}
    ]
}

print(db.covid_unified_data.count_documents(consulta))

540606


In [11]:
from pymongo.errors import BulkWriteError

db.mexico_city_covid_data.create_index("ID_REGISTRO", unique=True)

batch = []
batch_size = 10000
total_incertados = 0

filtro = {
    "ENTIDAD_UM": "09",
    "ENTIDAD_RES": {"$ne": "09"},
    "ENTIDAD_NAC": {"$ne": "09"}
}

cursor = db.covid_unified_data.find(filtro)


for doc in cursor:
    batch.append(doc)

    if len(batch) >= batch_size:
        try:
            db.mexico_city_covid_data.insert_many(batch, ordered=False)
            total_incertados += len(batch)
        except BulkWriteError:
            pass
        batch = []

if batch:
    try:
        db.mexico_city_covid_data.insert_many(batch, ordered=False)
        total_incertados += len(batch)
    except BulkWriteError:
        pass


total_final = db.mexico_city_covid_data.count_documents(filtro)
print(total_final)
print(f"Total de registros insertados: {total_incertados}")


540606
Total de registros insertados: 540606


In [ ]:
print(db.mexico_city_covid_data.count_documents({}))

7840211
